In [0]:
from pyspark.sql.functions import col,trim,lower,when,current_timestamp,row_number
from pyspark.sql.window import Window

bronze_df = spark.read.table("`reconciliation-platform`.bronze.order_items")

cleaned_df = (bronze_df
    .withColumn("order_item_id",trim(col("order_item_id")))
    .withColumn("order_id",trim(col("order_id")))
    .withColumn("product_id",trim(col("product_id")))
    .withColumn("category",lower(trim(col("category"))))
    .withColumn("quantity",col("quantity").cast("int"))
    .withColumn("unit_price",col("unit_price").cast("double"))
    .withColumn("item_discount",col("item_discount").cast("double"))
    .withColumn("item_total",col("item_total").cast("double"))
    .withColumn("updated_at",col("updated_at").cast("timestamp"))
    .withColumn("category",when(col("category").isNull() | (col("category") == ""),"unknown").otherwise(col("category")))
    .withColumn("item_discount",when(col("item_discount").isNull(),0.0).otherwise(col("item_discount")))
    .filter(col("order_item_id").isNotNull())
    .filter(col("order_item_id") != "")
    .filter(col("order_id").isNotNull())
    .filter(col("order_id") != "")
    .filter(col("product_id").isNotNull())
    .filter(col("product_id") != "")
    .filter(col("quantity") > 0)
    .filter(col("unit_price") >= 0)
    .filter(col("item_discount") >= 0)
    .withColumn(
        "item_total",
        when(
            col("item_total").isNull(),
            (col("quantity") * col("unit_price")) - col("item_discount")
        ).otherwise(col("item_total"))
    )
)

window_spec = Window.partitionBy("order_item_id").orderBy(
    col("updated_at").desc_nulls_last(),
    col("_ingestion_timestamp").desc()
)

silver_df = (cleaned_df
    .withColumn("_row_number",row_number().over(window_spec))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
    .withColumn("_silver_processed_at",current_timestamp())
)

(silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("`reconciliation-platform`.silver.order_items"))